# Churn Modeling 07-4 - Lightweight F1 Optimization

`07-3`에서는 `scoring='roc_auc'`로 hyperparameter search를 수행했기 때문에 ROC AUC는 0.90 이상으로 높았지만, F1은 0.68~0.70 근처에서 머물렀다.

이번 노트북의 목표는 다르다.

- CV search 단계에서 **F1을 직접 최적화**한다.
- ROC AUC와 PR AUC는 함께 기록하되, refit 기준은 F1로 둔다.
- 학습 시간이 너무 길어지지 않도록 `07-3`보다 후보 모델과 search 반복 수를 줄인다.
- valid set에서 threshold를 한 번 더 튜닝하고, test set은 마지막 확인에만 사용한다.


## 피쳐 설명

이 노트북은 `07-3`의 compact feature set을 유지하되, search의 refit 기준을 F1로 두는 가벼운 실험이다. feature engineering 범위는 넓히지 않고, F1 최적화 방향이 실제 test 성능으로 이어지는지 확인한다.

| 구분 | 피쳐명 | 설명 |
| --- | --- | --- |
| 사용 강도 | `watch_time_per_session` | 세션당 평균 시청 시간 |
| 계정 연령 대비 사용 | `watch_time_per_account_age` | 계정 연령 대비 주당 평균 시청 시간 |
| 계정 연령 대비 사용 | `session_per_account_age` | 계정 연령 대비 접속 세션 수 |
| 완료 행동 | `completion_per_watch_time` | 시청 시간 대비 콘텐츠 완료율 |
| 추천 반응 | `recommendation_completion_gap` | 추천 클릭률과 완료율의 차이 |
| 비활성 | `inactive_ratio_to_account_age` | 계정 연령 대비 미접속 기간 비율 |
| Engagement | `engagement_score` | 시청 시간, 시청 세션, 완료율, 추천 반응, 최근 접속성을 p95 기준으로 정규화해 결합한 점수 |
| 사용량 flag | `low_watch_flag` | 시청 시간이 train 1사분위수 이하인 사용자 flag |
| 사용량 flag | `low_session_flag` | 시청 세션 수가 train 1사분위수 이하인 사용자 flag |
| 완료 flag | `low_completion_flag` | 완료율이 train 1사분위수 이하인 사용자 flag |
| 추천 flag | `low_reco_flag` | 추천 클릭률이 train 1사분위수 이하인 사용자 flag |
| 비활성 flag | `high_inactivity_flag` | 미접속 기간이 train 3사분위수 이상인 사용자 flag |
| 결합 위험 | `inactive_low_watch_flag` | 높은 비활성과 낮은 시청량이 동시에 나타나는 사용자 flag |
| 결합 위험 | `low_interest_flag` | 낮은 추천 반응과 낮은 완료율이 동시에 나타나는 사용자 flag |
| 범주 조합 | `basic_mobile_flag` | Basic 요금제와 Mobile 중심 사용 조합 |
| 위험 집계 | `high_risk_behavior_count` | compact 위험 flag들의 합계 |

주의: `07-4`는 feature set을 새로 넓히는 노트북이 아니라, 같은 compact feature set에서 F1 기준 search가 효과적인지 확인하는 노트북이다.

## 1. 라이브러리 로드

학습 시간을 줄이기 위해 search 반복 수를 작게 둔다. `N_ITER_FAST`를 늘리면 성능 탐색은 넓어지지만 실행 시간이 길어진다.


In [1]:
from pathlib import Path
import warnings

import matplotlib.font_manager as fm
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import randint, loguniform, uniform

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
N_SPLITS = 3
N_ITER_FAST = 6
SEARCH_N_JOBS = 1  # nested parallelism을 피해서 과도한 CPU 경쟁을 줄인다.
MODEL_N_JOBS = -1

font_candidates = ['AppleGothic', 'NanumGothic', 'Malgun Gothic']
available_fonts = {font.name for font in fm.fontManager.ttflist}
font_name = next((font for font in font_candidates if font in available_fonts), None)
if font_name:
    plt.rcParams['font.family'] = font_name
plt.rcParams['axes.unicode_minus'] = False
sns.set_theme(style='whitegrid')


def make_ohe(**kwargs):
    try:
        return OneHotEncoder(handle_unknown='ignore', sparse_output=False, **kwargs)
    except TypeError:
        return OneHotEncoder(handle_unknown='ignore', sparse=False, **kwargs)


## 2. 데이터 로드와 80 / 10 / 10 Split

`07-3`과 동일하게 train 80%, valid 10%, test 10%를 사용한다.

- Train: F1 기준 hyperparameter search
- Valid: threshold tuning과 후보 선택
- Test: 최종 성능 확인


In [2]:
data_candidates = [
    Path('data/user_behavior_50000/netflix_user_behavior_churn_50000v2.csv'),
    Path('../data/user_behavior_50000/netflix_user_behavior_churn_50000v2.csv'),
]
data_path = next(path for path in data_candidates if path.exists())

df = pd.read_csv(data_path)
target_col = 'churned' if 'churned' in df.columns else 'churn'

X = df.drop(columns=['user_id', target_col])
y = df[target_col]

X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X,
    y,
    test_size=0.10,
    stratify=y,
    random_state=RANDOM_STATE,
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid,
    y_train_valid,
    test_size=1/9,
    stratify=y_train_valid,
    random_state=RANDOM_STATE,
)

print('Data path:', data_path)
print('Target column:', target_col)
print('Train shape:', X_train.shape, 'Valid shape:', X_valid.shape, 'Test shape:', X_test.shape)
print('Train churn ratio:', round(y_train.mean(), 4))
print('Valid churn ratio:', round(y_valid.mean(), 4))
print('Test churn ratio:', round(y_test.mean(), 4))
X_train.head()


Data path: ../data/user_behavior_50000/netflix_user_behavior_churn_50000v2.csv
Target column: churned
Train shape: (40000, 18) Valid shape: (5000, 18) Test shape: (5000, 18)
Train churn ratio: 0.2093
Valid churn ratio: 0.2092
Test churn ratio: 0.2092


,age,gender,region,subscription_type,payment_method,primary_device,account_age_months,favorite_genre,time_of_day,recommendation_source,session_count,avg_watch_time_minutes_per_week,watch_sessions_per_week,completion_rate,avg_rating_given,app_rating,recommendation_click_rate,days_since_last_login
33871,21,Male,Africa,Basic,Debit Card,Smart TV,7,Drama,Evening,Trending,1,120,2,72,5,4,12,2
36832,30,Male,South America,Basic,Debit Card,Mobile,40,Documentary,Afternoon,Algorithm,1,141,4,74,3,3,54,33
22002,19,Male,North America,Basic,Paypal,Smart TV,54,Action,Evening,Friend,1,266,7,89,3,4,41,6
11820,43,Female,South America,Standard,Credit Card,Mobile,74,Action,Evening,Friend,3,142,5,66,3,4,51,51
30475,20,Female,Europe,Basic,Credit Card,Smart TV,35,Drama,Night,Search,1,820,19,82,5,5,35,5


## 3. Compact Feature Engineering

`07-4`에서도 `07-3`의 compact feature만 비교한다. 많은 feature를 추가하는 것이 아니라, F1 개선 가능성이 있는 소수 행동 신호만 사용한다.


In [3]:
class CompactChurnFeatureEngineer(BaseEstimator, TransformerMixin):
    """F1 최적화 실험용 compact feature transformer."""

    def __init__(self, eps=1e-6):
        self.eps = eps

    def fit(self, X, y=None):
        X_fit = X.copy()
        self.watch_p95_ = max(X_fit['avg_watch_time_minutes_per_week'].quantile(0.95), self.eps)
        self.session_p95_ = max(X_fit['watch_sessions_per_week'].quantile(0.95), self.eps)
        self.inactive_p95_ = max(X_fit['days_since_last_login'].quantile(0.95), self.eps)
        self.watch_low_q_ = X_fit['avg_watch_time_minutes_per_week'].quantile(0.25)
        self.session_low_q_ = X_fit['watch_sessions_per_week'].quantile(0.25)
        self.completion_low_q_ = X_fit['completion_rate'].quantile(0.25)
        self.reco_low_q_ = X_fit['recommendation_click_rate'].quantile(0.25)
        self.inactive_high_q_ = X_fit['days_since_last_login'].quantile(0.75)
        return self

    def transform(self, X):
        X_fe = X.copy()
        eps = self.eps

        X_fe['watch_time_per_session'] = X_fe['avg_watch_time_minutes_per_week'] / (X_fe['watch_sessions_per_week'] + eps)
        X_fe['watch_time_per_account_age'] = X_fe['avg_watch_time_minutes_per_week'] / (X_fe['account_age_months'] + 1)
        X_fe['session_per_account_age'] = X_fe['session_count'] / (X_fe['account_age_months'] + 1)
        X_fe['completion_per_watch_time'] = X_fe['completion_rate'] / (X_fe['avg_watch_time_minutes_per_week'] + 1)
        X_fe['recommendation_completion_gap'] = X_fe['recommendation_click_rate'] - X_fe['completion_rate']
        X_fe['inactive_ratio_to_account_age'] = X_fe['days_since_last_login'] / (X_fe['account_age_months'] * 30 + 1)

        watch_norm = np.clip(X_fe['avg_watch_time_minutes_per_week'] / self.watch_p95_, 0, 1)
        session_norm = np.clip(X_fe['watch_sessions_per_week'] / self.session_p95_, 0, 1)
        completion_norm = np.clip(X_fe['completion_rate'] / 100, 0, 1)
        reco_norm = np.clip(X_fe['recommendation_click_rate'] / 100, 0, 1)
        recency_norm = 1 - np.clip(X_fe['days_since_last_login'] / self.inactive_p95_, 0, 1)
        X_fe['engagement_score'] = 100 * (
            0.25 * watch_norm
            + 0.20 * session_norm
            + 0.25 * completion_norm
            + 0.15 * reco_norm
            + 0.15 * recency_norm
        )

        X_fe['low_watch_flag'] = (X_fe['avg_watch_time_minutes_per_week'] <= self.watch_low_q_).astype(int)
        X_fe['low_session_flag'] = (X_fe['watch_sessions_per_week'] <= self.session_low_q_).astype(int)
        X_fe['low_completion_flag'] = (X_fe['completion_rate'] <= self.completion_low_q_).astype(int)
        X_fe['low_reco_flag'] = (X_fe['recommendation_click_rate'] <= self.reco_low_q_).astype(int)
        X_fe['high_inactivity_flag'] = (X_fe['days_since_last_login'] >= self.inactive_high_q_).astype(int)
        X_fe['inactive_low_watch_flag'] = ((X_fe['high_inactivity_flag'] == 1) & (X_fe['low_watch_flag'] == 1)).astype(int)
        X_fe['low_interest_flag'] = ((X_fe['low_reco_flag'] == 1) & (X_fe['low_completion_flag'] == 1)).astype(int)
        X_fe['basic_mobile_flag'] = (
            (X_fe['subscription_type'].astype(str) == 'Basic')
            & (X_fe['primary_device'].astype(str) == 'Mobile')
        ).astype(int)

        risk_cols = [
            'low_watch_flag',
            'low_session_flag',
            'low_completion_flag',
            'low_reco_flag',
            'high_inactivity_flag',
            'inactive_low_watch_flag',
            'low_interest_flag',
            'basic_mobile_flag',
        ]
        X_fe['high_risk_behavior_count'] = X_fe[risk_cols].sum(axis=1)
        return X_fe


X_train_compact = CompactChurnFeatureEngineer().fit_transform(X_train)
added_cols = sorted(set(X_train_compact.columns) - set(X_train.columns))
print('Added feature count:', len(added_cols))
added_cols


Added feature count: 16


['basic_mobile_flag',
 'completion_per_watch_time',
 'engagement_score',
 'high_inactivity_flag',
 'high_risk_behavior_count',
 'inactive_low_watch_flag',
 'inactive_ratio_to_account_age',
 'low_completion_flag',
 'low_interest_flag',
 'low_reco_flag',
 'low_session_flag',
 'low_watch_flag',
 'recommendation_completion_gap',
 'session_per_account_age',
 'watch_time_per_account_age',
 'watch_time_per_session']

## 4. Pipeline과 평가 함수

이번 실험에서는 CV search의 `refit` 기준을 `f1`로 둔다. 이후 valid set에서 threshold를 다시 조정해 F1을 한 번 더 끌어올린다.


In [4]:
def build_preprocessor(X_schema):
    numeric_features = X_schema.select_dtypes(include='number').columns.tolist()
    categorical_features = X_schema.select_dtypes(include=['object', 'string', 'category']).columns.tolist()
    return ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_features),
            ('cat', make_ohe(), categorical_features),
        ],
        remainder='drop',
        verbose_feature_names_out=False,
    )


def build_pipeline(X_train_schema, estimator, feature_set='original'):
    steps = []
    if feature_set == 'compact_fe':
        fe = CompactChurnFeatureEngineer()
        X_schema = fe.fit_transform(X_train_schema)
        steps.append(('feature_engineering', fe))
    elif feature_set == 'original':
        X_schema = X_train_schema.copy()
    else:
        raise ValueError(f'Unknown feature_set: {feature_set}')

    steps.extend([
        ('preprocess', build_preprocessor(X_schema)),
        ('model', clone(estimator)),
    ])
    return Pipeline(steps)


def get_scores(model, X_eval):
    if hasattr(model, 'predict_proba'):
        return model.predict_proba(X_eval)[:, 1]
    raw_scores = model.decision_function(X_eval)
    return (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())


def tune_threshold_for_f1(y_true, y_score, thresholds=np.arange(0.05, 0.951, 0.005)):
    rows = []
    for threshold in thresholds:
        y_pred = (y_score >= threshold).astype(int)
        rows.append({
            'threshold': threshold,
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
        })
    threshold_df = pd.DataFrame(rows)
    best_row = threshold_df.loc[threshold_df['f1'].idxmax()]
    return best_row, threshold_df


def evaluate_auc_f1(y_true, y_score, threshold=0.5):
    y_pred = (y_score >= threshold).astype(int)
    return {
        'roc_auc': roc_auc_score(y_true, y_score),
        'pr_auc': average_precision_score(y_true, y_score),
        'threshold': threshold,
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'confusion_matrix': confusion_matrix(y_true, y_pred),
    }


## 5. Lightweight Search Space

`07-3`보다 가볍게 구성한다.

- RandomForest / ExtraTrees는 제외한다. `07-3`에서 F1 우위가 크지 않았고 시간이 오래 걸렸다.
- Logistic Regression은 빠른 기준선으로 유지한다.
- LightGBM, XGBoost, CatBoost는 boosting 후보로 유지하되 `n_iter`를 줄인다.
- `RandomizedSearchCV(n_jobs=1)`로 두고, 모델 내부 병렬만 사용해 nested parallelism을 줄인다.


In [5]:
def available_f1_model_spaces(y_train):
    spaces = {}

    spaces['LogisticRegression'] = {
        'estimator': LogisticRegression(max_iter=3000, random_state=RANDOM_STATE),
        'params': {
            'model__C': loguniform(0.03, 30),
            'model__class_weight': [None, 'balanced'],
        },
        'n_iter': 6,
    }

    try:
        from lightgbm import LGBMClassifier
        spaces['LightGBM'] = {
            'estimator': LGBMClassifier(random_state=RANDOM_STATE, n_jobs=MODEL_N_JOBS, verbose=-1),
            'params': {
                'model__n_estimators': randint(200, 700),
                'model__learning_rate': loguniform(0.015, 0.12),
                'model__num_leaves': randint(16, 80),
                'model__min_child_samples': randint(10, 100),
                'model__subsample': uniform(0.70, 0.30),
                'model__colsample_bytree': uniform(0.70, 0.30),
                'model__class_weight': [None, 'balanced'],
            },
            'n_iter': N_ITER_FAST,
        }
    except Exception as exc:
        print('[skip] LightGBM unavailable:', exc)

    try:
        from xgboost import XGBClassifier
        neg, pos = np.bincount(y_train)
        spaces['XGBoost'] = {
            'estimator': XGBClassifier(
                eval_metric='logloss',
                tree_method='hist',
                random_state=RANDOM_STATE,
                n_jobs=MODEL_N_JOBS,
            ),
            'params': {
                'model__n_estimators': randint(200, 650),
                'model__learning_rate': loguniform(0.015, 0.12),
                'model__max_depth': randint(3, 7),
                'model__min_child_weight': randint(1, 10),
                'model__subsample': uniform(0.70, 0.30),
                'model__colsample_bytree': uniform(0.70, 0.30),
                'model__scale_pos_weight': [1, neg / pos, np.sqrt(neg / pos)],
            },
            'n_iter': N_ITER_FAST,
        }
    except Exception as exc:
        print('[skip] XGBoost unavailable:', exc)

    try:
        from catboost import CatBoostClassifier
        spaces['CatBoost'] = {
            'estimator': CatBoostClassifier(
                loss_function='Logloss',
                eval_metric='F1',
                random_seed=RANDOM_STATE,
                verbose=False,
                allow_writing_files=False,
                thread_count=MODEL_N_JOBS,
            ),
            'params': {
                'model__iterations': randint(200, 600),
                'model__learning_rate': loguniform(0.015, 0.12),
                'model__depth': randint(4, 8),
                'model__l2_leaf_reg': loguniform(1, 15),
                'model__auto_class_weights': [None, 'Balanced', 'SqrtBalanced'],
            },
            'n_iter': N_ITER_FAST,
        }
    except Exception as exc:
        print('[skip] CatBoost unavailable:', exc)

    return spaces


model_spaces = available_f1_model_spaces(y_train)
print('Model candidates:', list(model_spaces.keys()))


Model candidates: ['LogisticRegression', 'LightGBM', 'XGBoost', 'CatBoost']


## 6. F1 기준 Hyperparameter Search

이전 `07-3`과 가장 큰 차이는 여기다.

```python
scoring={...}
refit='f1'
```

이렇게 설정하면 CV에서 여러 지표를 기록하되, 최종 best estimator는 F1 기준으로 선택된다.


In [6]:
cv = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)
feature_sets = ['original', 'compact_fe']
scoring = {
    'f1': 'f1',
    'roc_auc': 'roc_auc',
    'pr_auc': 'average_precision',
}

search_rows = []
fitted_candidates = {}
valid_scores = {}
valid_threshold_tables = {}

for feature_set in feature_sets:
    for model_name, spec in model_spaces.items():
        run_name = f'{model_name}__{feature_set}'
        total_fits = spec['n_iter'] * N_SPLITS
        print(f'[search] {run_name} | n_iter={spec["n_iter"]}, total_fits={total_fits}', flush=True)

        pipe = build_pipeline(X_train, spec['estimator'], feature_set=feature_set)
        search = RandomizedSearchCV(
            estimator=pipe,
            param_distributions=spec['params'],
            n_iter=spec['n_iter'],
            scoring=scoring,
            refit='f1',
            cv=cv,
            n_jobs=SEARCH_N_JOBS,
            random_state=RANDOM_STATE,
            verbose=1,
        )
        search.fit(X_train, y_train)

        best_model = search.best_estimator_
        y_valid_score = get_scores(best_model, X_valid)
        valid_at_05 = evaluate_auc_f1(y_valid, y_valid_score, threshold=0.5)
        best_threshold_row, threshold_df = tune_threshold_for_f1(y_valid, y_valid_score)
        valid_tuned = evaluate_auc_f1(y_valid, y_valid_score, threshold=float(best_threshold_row['threshold']))

        fitted_candidates[run_name] = best_model
        valid_scores[run_name] = y_valid_score
        valid_threshold_tables[run_name] = threshold_df

        search_rows.append({
            'run_name': run_name,
            'model': model_name,
            'feature_set': feature_set,
            'best_cv_f1': search.best_score_,
            'mean_cv_roc_auc_at_best_index': search.cv_results_['mean_test_roc_auc'][search.best_index_],
            'mean_cv_pr_auc_at_best_index': search.cv_results_['mean_test_pr_auc'][search.best_index_],
            'valid_roc_auc': valid_at_05['roc_auc'],
            'valid_pr_auc': valid_at_05['pr_auc'],
            'valid_f1_at_0_5': valid_at_05['f1'],
            'best_valid_threshold': float(best_threshold_row['threshold']),
            'valid_best_f1': valid_tuned['f1'],
            'valid_precision_at_best_f1': valid_tuned['precision'],
            'valid_recall_at_best_f1': valid_tuned['recall'],
            'best_params': search.best_params_,
        })

        print(
            f"[done] {run_name} | "
            f"CV F1={search.best_score_:.4f}, "
            f"Valid F1@0.5={valid_at_05['f1']:.4f}, "
            f"Valid best F1={valid_tuned['f1']:.4f}, "
            f"threshold={float(best_threshold_row['threshold']):.3f}, "
            f"Valid ROC AUC={valid_at_05['roc_auc']:.4f}",
            flush=True,
        )

search_result = pd.DataFrame(search_rows).sort_values(
    ['valid_best_f1', 'valid_roc_auc', 'best_cv_f1'],
    ascending=False,
).reset_index(drop=True)

search_result[[
    'run_name',
    'best_cv_f1',
    'mean_cv_roc_auc_at_best_index',
    'mean_cv_pr_auc_at_best_index',
    'valid_roc_auc',
    'valid_pr_auc',
    'valid_f1_at_0_5',
    'best_valid_threshold',
    'valid_best_f1',
    'valid_precision_at_best_f1',
    'valid_recall_at_best_f1',
]]


[search] LogisticRegression__original | n_iter=6, total_fits=18
Fitting 3 folds for each of 6 candidates, totalling 18 fits
[done] LogisticRegression__original | CV F1=0.6735, Valid F1@0.5=0.6790, Valid best F1=0.7111, threshold=0.640, Valid ROC AUC=0.9166
[search] LightGBM__original | n_iter=6, total_fits=18
Fitting 3 folds for each of 6 candidates, totalling 18 fits
[done] LightGBM__original | CV F1=0.6815, Valid F1@0.5=0.6882, Valid best F1=0.7000, threshold=0.545, Valid ROC AUC=0.9098
[search] XGBoost__original | n_iter=6, total_fits=18
Fitting 3 folds for each of 6 candidates, totalling 18 fits
[done] XGBoost__original | CV F1=0.6899, Valid F1@0.5=0.7006, Valid best F1=0.7009, threshold=0.520, Valid ROC AUC=0.9139
[search] CatBoost__original | n_iter=6, total_fits=18
Fitting 3 folds for each of 6 candidates, totalling 18 fits
[done] CatBoost__original | CV F1=0.6854, Valid F1@0.5=0.6896, Valid best F1=0.6960, threshold=0.435, Valid ROC AUC=0.9106
[search] LogisticRegression__compa

,run_name,best_cv_f1,mean_cv_roc_auc_at_best_index,mean_cv_pr_auc_at_best_index,valid_roc_auc,valid_pr_auc,valid_f1_at_0_5,best_valid_threshold,valid_best_f1,valid_precision_at_best_f1,valid_recall_at_best_f1
0,LogisticRegression__original,0.673544,0.910066,0.761427,0.916593,0.766233,0.678998,0.640,0.711052,0.663629,0.765774
1,LogisticRegression__compact_fe,0.673168,0.909846,0.760856,0.916289,0.766179,0.679739,0.640,0.710597,0.665000,0.762906
2,XGBoost__compact_fe,0.688690,0.906701,0.755169,0.913019,0.760839,0.700364,0.485,0.701880,0.659933,0.749522
3,XGBoost__original,0.689886,0.907326,0.755571,0.913893,0.760340,0.700637,0.520,0.700926,0.679533,0.723709
4,LightGBM__original,0.681535,0.902027,0.746120,0.909767,0.752638,0.688172,0.545,0.699957,0.636648,0.777247
5,LightGBM__compact_fe,0.680906,0.902300,0.746521,0.908981,0.752997,0.688345,0.575,0.699332,0.654712,0.750478
6,CatBoost__original,0.685393,0.903053,0.748244,0.910566,0.752729,0.689623,0.435,0.695996,0.644662,0.756214
7,CatBoost__compact_fe,0.681639,0.902909,0.747800,0.908656,0.749232,0.686888,0.480,0.691991,0.663740,0.722753


## 7. Test 평가

valid F1 기준 상위 후보만 test에서 확인한다. test threshold는 valid에서 선택한 threshold를 그대로 사용한다.


In [7]:
top_k = min(6, len(search_result))
test_rows = []
test_confusions = {}
test_scores = {}

for _, candidate in search_result.head(top_k).iterrows():
    run_name = candidate['run_name']
    model = fitted_candidates[run_name]
    threshold = float(candidate['best_valid_threshold'])

    y_test_score = get_scores(model, X_test)
    test_at_05 = evaluate_auc_f1(y_test, y_test_score, threshold=0.5)
    test_tuned = evaluate_auc_f1(y_test, y_test_score, threshold=threshold)

    test_scores[run_name] = y_test_score
    test_confusions[run_name] = test_tuned['confusion_matrix']

    test_rows.append({
        'run_name': run_name,
        'model': candidate['model'],
        'feature_set': candidate['feature_set'],
        'best_cv_f1': candidate['best_cv_f1'],
        'valid_best_f1': candidate['valid_best_f1'],
        'valid_selected_threshold': threshold,
        'test_roc_auc': test_at_05['roc_auc'],
        'test_pr_auc': test_at_05['pr_auc'],
        'test_f1_at_0_5': test_at_05['f1'],
        'test_f1_at_valid_threshold': test_tuned['f1'],
        'test_precision_at_valid_threshold': test_tuned['precision'],
        'test_recall_at_valid_threshold': test_tuned['recall'],
    })

test_result = pd.DataFrame(test_rows).sort_values(
    ['test_f1_at_valid_threshold', 'test_roc_auc'],
    ascending=False,
).reset_index(drop=True)

test_result


,run_name,model,feature_set,best_cv_f1,valid_best_f1,valid_selected_threshold,test_roc_auc,test_pr_auc,test_f1_at_0_5,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold
0,LogisticRegression__original,LogisticRegression,original,0.673544,0.711052,0.640,0.902673,0.752783,0.668222,0.683080,0.642256,0.729446
1,LogisticRegression__compact_fe,LogisticRegression,compact_fe,0.673168,0.710597,0.640,0.902080,0.751943,0.665370,0.680108,0.639966,0.725621
2,XGBoost__compact_fe,XGBoost,compact_fe,0.688690,0.701880,0.485,0.899587,0.752744,0.679630,0.679142,0.649782,0.711281
3,LightGBM__original,LightGBM,original,0.681535,0.699957,0.545,0.896836,0.746564,0.674507,0.675687,0.621687,0.739962
4,XGBoost__original,XGBoost,original,0.689886,0.700926,0.520,0.900418,0.753668,0.677809,0.674847,0.666356,0.683556
5,LightGBM__compact_fe,LightGBM,compact_fe,0.680906,0.699332,0.575,0.895342,0.743019,0.674024,0.672390,0.637532,0.711281


## 8. 최종 F1 후보 확인

이번 노트북에서는 `test_f1_at_valid_threshold`가 가장 높은 후보를 우선한다. AUC는 보조 확인 지표다.


In [8]:
best_f1_row = test_result.loc[test_result['test_f1_at_valid_threshold'].idxmax()]
best_name = best_f1_row['run_name']
best_cm = test_confusions[best_name]

print('Best F1 candidate')
display(best_f1_row.to_frame().T)
print('Confusion matrix at valid-selected threshold')
print(best_cm)

test_result


Best F1 candidate


,run_name,model,feature_set,best_cv_f1,valid_best_f1,valid_selected_threshold,test_roc_auc,test_pr_auc,test_f1_at_0_5,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold
0,LogisticRegression__original,LogisticRegression,original,0.673544,0.711052,0.64,0.902673,0.752783,0.668222,0.68308,0.642256,0.729446


Confusion matrix at valid-selected threshold
[[3529  425]
 [ 283  763]]


,run_name,model,feature_set,best_cv_f1,valid_best_f1,valid_selected_threshold,test_roc_auc,test_pr_auc,test_f1_at_0_5,test_f1_at_valid_threshold,test_precision_at_valid_threshold,test_recall_at_valid_threshold
0,LogisticRegression__original,LogisticRegression,original,0.673544,0.711052,0.640,0.902673,0.752783,0.668222,0.683080,0.642256,0.729446
1,LogisticRegression__compact_fe,LogisticRegression,compact_fe,0.673168,0.710597,0.640,0.902080,0.751943,0.665370,0.680108,0.639966,0.725621
2,XGBoost__compact_fe,XGBoost,compact_fe,0.688690,0.701880,0.485,0.899587,0.752744,0.679630,0.679142,0.649782,0.711281
3,LightGBM__original,LightGBM,original,0.681535,0.699957,0.545,0.896836,0.746564,0.674507,0.675687,0.621687,0.739962
4,XGBoost__original,XGBoost,original,0.689886,0.700926,0.520,0.900418,0.753668,0.677809,0.674847,0.666356,0.683556
5,LightGBM__compact_fe,LightGBM,compact_fe,0.680906,0.699332,0.575,0.895342,0.743019,0.674024,0.672390,0.637532,0.711281


## 10. 실험 결과 해석

`07-4`의 목적은 `07-3`과 다르게 ROC AUC가 아니라 **F1 score를 직접 높이는 것**이었다. 그래서 `RandomizedSearchCV`에서 multi-metric scoring을 사용하되, 최종 best estimator는 `refit='f1'` 기준으로 선택했다.

실험 결과, 방향 자체는 맞았지만 test set 기준으로는 `07-3`의 최고 F1을 넘지 못했다.

- `07-4` 최고 후보: `LogisticRegression__original`
- `07-4` test F1: 0.6831
- `07-4` test ROC AUC: 0.9027
- `07-4` valid-selected threshold: 0.640
- `07-3` 최고 F1 후보: `CatBoost__compact_fe`
- `07-3` 최고 test F1: 0.6856

즉 `07-4`는 F1 기준 search를 적용했지만, 최종 test F1은 `07-3`보다 약간 낮았다. 따라서 현재까지 F1 기준 최고 모델은 여전히 `07-3`의 `CatBoost__compact_fe`다.


## 11. 왜 F1 Search가 더 좋아지지 않았나

`07-4`는 F1을 직접 최적화했지만, search를 빠르게 끝내기 위해 `N_ITER_FAST = 6`으로 제한했다. 전체 실행 시간은 줄었지만, CatBoost나 XGBoost의 좋은 조합을 충분히 탐색하지 못했을 가능성이 있다.

또한 valid set에서는 `LogisticRegression__original`이 가장 좋아 보였다.

- valid best F1: 0.7111
- test F1: 0.6831

valid와 test 사이에 약간의 성능 차이가 발생했다. 이는 threshold tuning이 valid set에 맞춰졌기 때문에 자연스럽게 생길 수 있는 차이다. 중요한 점은 이 차이가 모델 실패라기보다는, F1 threshold가 데이터 split에 민감하다는 신호라는 것이다.

ROC AUC는 여전히 0.9027로 높다. 즉 ranking 성능은 유지됐지만, threshold 기반 F1은 기대만큼 올라가지 않았다.


## 12. Feature Engineering 관점 해석

`07-4`에서도 `original`과 `compact_fe`를 비교했다. 결과적으로 이번 실험에서는 compact feature가 F1 개선으로 이어지지 않았다.

주요 비교는 다음과 같다.

- `LogisticRegression__original`: test F1 0.6831
- `LogisticRegression__compact_fe`: test F1 0.6801
- `XGBoost__compact_fe`: test F1 0.6791
- `XGBoost__original`: test F1 0.6748

XGBoost에서는 compact feature가 original보다 조금 낫지만, 전체 최고 후보를 넘지는 못했다. Logistic Regression에서는 original이 compact feature보다 좋았다.

따라서 현재 데이터에서는 원본 피처 자체가 꽤 강하고, compact feature는 특정 모델에서는 도움이 될 수 있지만 일관된 성능 개선 근거는 아직 부족하다.


## 13. 07-4 최종 결론

`07-4`는 F1 최적화 방향을 확인하기 위한 가벼운 실험으로는 의미가 있다. 하지만 최종 성능 기준으로는 `07-3`을 넘지 못했다.

정리하면 다음과 같다.

1. `refit='f1'`로 search 기준을 바꾼 것은 올바른 실험 방향이다.
2. 다만 `N_ITER_FAST = 6`으로 제한했기 때문에 탐색 깊이가 부족했을 수 있다.
3. test F1 최고값은 0.6831로, `07-3`의 0.6856보다 낮다.
4. 현재까지 F1 기준 최고 후보는 `07-3`의 `CatBoost__compact_fe`다.
5. 현재까지 단순성과 안정성 기준으로는 `LogisticRegression__original`이 여전히 강하다.

다음 단계에서 F1을 더 밀어붙인다면 전체 모델을 다시 길게 돌리기보다, 가능성이 있었던 `CatBoost`와 `XGBoost`만 대상으로 `n_iter`를 10~15 정도로 늘리는 것이 합리적이다.


## 14. 해석 가이드

`07-4`는 `07-3`과 비교해서 다음 질문에 답하기 위한 실험이다.

> Search 자체를 F1 기준으로 바꾸면 test F1이 실제로 올라가는가?

해석 기준은 다음과 같다.

- `test_f1_at_valid_threshold`가 `07-3`의 최고 F1인 0.6856보다 높으면 F1 기준 search가 효과가 있었다고 볼 수 있다.
- F1은 올랐지만 ROC AUC가 크게 떨어지면, ranking 성능을 희생한 trade-off가 발생한 것이다.
- F1과 ROC AUC가 둘 다 비슷하거나 좋아지면 `07-4` 후보가 더 적절하다.
- compact feature가 original보다 좋을 때만 feature engineering을 최종 후보에 포함한다.

주의할 점은, CV의 `f1`은 기본 threshold 0.5 기준이라는 것이다. 그래서 valid set threshold tuning을 별도로 수행한다. 최종 test 평가는 valid에서 선택한 threshold를 그대로 사용한다.
